In [ ]:

import numpy as np
import os
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import StepLR
import matplotlib.pyplot as plt
import mnist1d
import random

In [ ]:
args = mnist1d.data.get_dataset_args()
data = mnist1d.data.get_dataset(args, path='./mnist1d_data.pkl', download=False, regenerate=False)

# The training and test input and outputs are in
# data['x'], data['y'], data['x_test'], and data['y_test']
print("Examples in training set: {}".format(len(data['y'])))
print("Examples in test set: {}".format(len(data['y_test'])))
print("Length of each example: {}".format(data['x'].shape[-1]))

Successfully loaded data from ./mnist1d_data.pkl
Examples in training set: 4000
Examples in test set: 1000
Length of each example: 40


In [ ]:
train_data_x = data['x'].transpose()
train_data_y = data['y']
val_data_x = data['x_test'].transpose()
val_data_y = data['y_test']
# Print out sizes
print("Train data: %d examples (columns), each of which has %d dimensions (rows)"%((train_data_x.shape[1],train_data_x.shape[0])))
print("Validation data: %d examples (columns), each of which has %d dimensions (rows)"%((val_data_x.shape[1],val_data_x.shape[0])))

Train data: 4000 examples (columns), each of which has 40 dimensions (rows)
Validation data: 1000 examples (columns), each of which has 40 dimensions (rows)


In [ ]:
D_i = 40
D_o = 10

# 1. Convolutional layer, (input=length 40 and 1 channel, kernel size 3, stride 2, padding="valid", 15 output channels )
# 2. ReLU
# 3. Convolutional layer, (input=length 19 and 15 channels, kernel size 3, stride 2, padding="valid", 15 output channels )
# 4. ReLU
# 5. Convolutional layer, (input=length 9 and 15 channels, kernel size 3, stride 2, padding="valid", 15 output channels)
# 6. ReLU
# 7. Flatten (converts 4x15) to length 60
# 8. Linear layer (input size = 60, output size = 10)


model = nn.Sequential(
        # 1. Convolutional layer (input=length 40 and 1 channel)
        # L_out = floor((40 - 3) / 2) + 1 = 19
        nn.Conv1d(
            in_channels=1,
            out_channels=15,
            kernel_size=3,
            stride=2,
            padding=0  # "valid" padding
        ),
        # 2. ReLU
        nn.ReLU(),
        
        # 3. Convolutional layer (input=length 19 and 15 channels)
        # L_out = floor((19 - 3) / 2) + 1 = 9
        nn.Conv1d(
            in_channels=15,
            out_channels=15,
            kernel_size=3,
            stride=2,
            padding=0
        ),
        # 4. ReLU
        nn.ReLU(),
        
        # 5. Convolutional layer (input=length 9 and 15 channels)
        # L_out = floor((9 - 3) / 2) + 1 = 4
        nn.Conv1d(
            in_channels=15,
            out_channels=15,
            kernel_size=3,
            stride=2,
            padding=0
        ),
        # 6. ReLU
        nn.ReLU(),
        
        # 7. Flatten (converts 4x15) to length 60
        # Input tensor shape here is (Batch, 15, 4). Flatten collapses 15 * 4 = 60.
        nn.Flatten(),
        
        # 8. Linear layer (input size = 60, output size = 10)
        nn.Linear(15 * 4, 10)
)

In [ ]:
def weights_init(layer_in):
  if(isinstance(layer_in, nn.Linear)):
    nn.init.kaiming_uniform_(layer_in.weight)
    layer_in.bias.data.fill_(0.0)

: 

In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
scheduler= StepLR(optimizer, step_size=20, gamma=0.5)

x_train = torch.tensor(train_data_x.transpose().astype("float32"))
y_train = torch.tensor(train_data_y.astype('long')).long()
x_val= torch.tensor(val_data_x.transpose().astype('float32'))
y_val = torch.tensor(val_data_y.astype('long')).long()

data_loader = DataLoader(
  TensorDataset(x_train, y_train),
  batch_size=100,
  shuffle=True,
  worker_init_fn=np.random.seed(1)
)

# Initialize model weights
model.apply(weights_init)

# loop over the dataset n_epoch times
n_epoch = 100
# store the loss and the % correct at each epoch
losses_train = np.zeros((n_epoch))
errors_train = np.zeros((n_epoch))
losses_val = np.zeros((n_epoch))
errors_val = np.zeros((n_epoch))

for epoch in range(n_epoch):
  # loop over batches
  for i, data in enumerate(data_loader):
    # retrieve inputs and labels for this batch
    x_batch, y_batch = data
    # zero the parameter gradients
    optimizer.zero_grad()
    # forward pass -- calculate model output
    pred = model(x_batch[:,None,:])
    # compute the loss
    loss = loss_function(pred, y_batch)
    # backward pass
    loss.backward()
    # SGD update
    optimizer.step()

  # Run whole dataset to get statistics -- normally wouldn't do this
  pred_train = model(x_train[:,None,:])
  pred_val = model(x_val[:,None,:])
  _, predicted_train_class = torch.max(pred_train.data, 1)
  _, predicted_val_class = torch.max(pred_val.data, 1)
  errors_train[epoch] = 100 - 100 * (predicted_train_class == y_train).float().sum() / len(y_train)
  errors_val[epoch]= 100 - 100 * (predicted_val_class == y_val).float().sum() / len(y_val)
  losses_train[epoch] = loss_function(pred_train, y_train).item()
  losses_val[epoch]= loss_function(pred_val, y_val).item()
  print(f'Epoch {epoch:5d}, train loss {losses_train[epoch]:.6f}, train error {errors_train[epoch]:3.2f},  val loss {losses_val[epoch]:.6f}, percent error {errors_val[epoch]:3.2f}')

  # tell scheduler to consider updating learning rate
  scheduler.step()

# Plot the results
fig, ax = plt.subplots()
ax.plot(errors_train,'r-',label='train')
ax.plot(errors_val,'b-',label='validation')
ax.set_ylim(0,100); ax.set_xlim(0,n_epoch)
ax.set_xlabel('Epoch'); ax.set_ylabel('Error')
ax.set_title('Part I: Validation Result %3.2f'%(errors_val[-1]))
ax.legend()
plt.show()

Epoch     0, train loss 2.134077, train error 79.78,  val loss 2.138374, percent error 81.40
Epoch     1, train loss 1.553204, train error 63.83,  val loss 1.558524, percent error 66.70
Epoch     2, train loss 1.325805, train error 54.28,  val loss 1.339528, percent error 58.10
Epoch     3, train loss 1.216758, train error 49.53,  val loss 1.241872, percent error 52.80
Epoch     4, train loss 1.121526, train error 45.60,  val loss 1.162530, percent error 47.10
Epoch     5, train loss 1.048365, train error 42.62,  val loss 1.087755, percent error 45.60
Epoch     6, train loss 0.904862, train error 35.95,  val loss 0.935871, percent error 37.90
Epoch     7, train loss 0.860198, train error 34.78,  val loss 0.911081, percent error 38.50
Epoch     8, train loss 0.782797, train error 31.18,  val loss 0.868309, percent error 35.90
Epoch     9, train loss 0.667718, train error 26.95,  val loss 0.746764, percent error 30.20
Epoch    10, train loss 0.626875, train error 23.95,  val loss 0.73422